# Dataset Inspection

In this section, we explore the dataset's features, size, and missing values to form our initial assumptions. This overview helps us identify potential correlations and plan how to handle missing data and outliers.

to do that we first import what we need pandas, sqlite3

In [ ]:
import sqlite3
import pandas as pd

Now let's see how the data in data/data.sqlite is made

In [ ]:
db = sqlite3.connect("../data/data.sqlite")

# Sample 5 rows from the Fires table
df_sample = pd.read_sql_query("SELECT * FROM Fires LIMIT 5;", db)

# List all features
print(df_sample.columns.tolist())
display(df_sample)

db.close()

We can start the preprocessing by dropping useless administrative features. We will keep only the following 13 relevant features:

- **Possible Targets**: `STAT_CAUSE_DESCR` (cause), `FIRE_SIZE`, `FIRE_SIZE_CLASS` (size)
- **Time**: `FIRE_YEAR`, `DISCOVERY_DATE`, `DISCOVERY_DOY`, `CONT_DATE`, `CONT_DOY`
- **Space**: `LATITUDE`, `LONGITUDE`, `STATE`, `COUNTY`, `OWNER_DESCR`

In [ ]:
db = sqlite3.connect("../data/data.sqlite")

# query to get the good features

cols = "FIRE_YEAR, DISCOVERY_DATE, DISCOVERY_DOY, STAT_CAUSE_DESCR, CONT_DATE, CONT_DOY, FIRE_SIZE, FIRE_SIZE_CLASS, LATITUDE, LONGITUDE, STATE, COUNTY, OWNER_DESCR"
df_sample = pd.read_sql_query(f"SELECT {cols} FROM Fires LIMIT 5;", db)

print(df_sample.columns.tolist())
display(df_sample)

db.close()

## Data Explanation

First, we can emphasize that `DISCOVERY_DATE` and `CONT_DATE` are encoded as Julian numbers that we have to decode. Secondly, `FIRE_SIZE_CLASS` is derived from `FIRE_SIZE`. We shouldn't use `FIRE_SIZE_CLASS` since it's less specific than `FIRE_SIZE`.

We can create new features or simplify existing ones. For instance, `DISCOVERY_DOY` can be useful if we treat it as a seasonal value. Assuming there are fewer fires in December than in summer, we can group the [1, 365] range into [1, 4] to represent seasons.

`STAT_CAUSE_DESCR` and `OWNER_DESCR` are text strings, but since they only have a small quantity of possible values, we can convert them using one-hot encoding.

We can also predict links between some features. For instance, it's clear that the duration of the fire ($\Delta = CONT\_DATE - DISCOVERY\_DATE$) is strongly linked to the size of the fire.

let's see what values are missing.